# Logistic regression — Boomerang vs Sticky Boomerang

Two experiments: binary (8 features, 2 true signals) and multinomial (6 features, 3 classes, dense coefficients).

**What to expect:**

- **Binary, plain Boomerang:** means track the sklearn MLE, stds match Laplace approximation. All P(=0) ≈ 0.
- **Binary, Sticky:** signals recovered, null coefs have P(=0) > 0.5.
- **Multiclass:** posterior means should correlate strongly with sklearn's fitted coefficients. Sticky with uniform small κ shrinks all coefficients — useful as a check that the sampler is working, less interesting as a sparsity story since the truth is dense.

**Warning sign:** if posterior stds ≈ `prior_std` everywhere, the sampler isn't exploring the likelihood.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

import os
os.chdir('../..')

from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler
from sazz.samplers.StickyAutomaticBoomerangSampler import StickyAutomaticBoomerangSampler
from sazz.models.glm import make_logistic_regression, make_kappa_vector_glm
from sazz.utils.sampling import resample_pdmp_path, resample_pdmp_path_sticky

## Helpers

In [ ]:
def run_both(target, kappa, N_skel=20_000, N_resample=50_000, burnin=0.1):
    """Run plain and sticky Boomerang on the same target, return resampled posteriors."""
    sampler = AutomaticBoomerangSampler(
        grad_target=target.grad_target, D=target.D, refresh_rate=1.0, thinning='pli',
    )
    sampler.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv)

    sampler_s = StickyAutomaticBoomerangSampler(
        grad_target=target.grad_target, D=target.D, refresh_rate=1.0,
        kappa=kappa, thinning='pli',
    )
    sampler_s.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv)

    r = sampler.sample(N=N_skel, diagnostics=False)
    rs = sampler_s.sample(N=N_skel, diagnostics=False)

    x_ref_np = target.x_ref.cpu().numpy()
    samples = resample_pdmp_path(
        r['positions'].cpu().numpy(), r['velocities'].cpu().numpy(),
        r['times'].cpu().numpy(), x_ref_np,
        N_resample=N_resample, burnin_frac=burnin,
    )
    samples_s = resample_pdmp_path_sticky(
        rs['positions'].cpu().numpy(), rs['velocities'].cpu().numpy(),
        rs['times'].cpu().numpy(), x_ref_np,
        N_resample=N_resample, burnin_frac=burnin,
    )
    return samples, samples_s


def calibration_plot(samples, samples_s, true_coefs, ref_coefs, labels,
                     is_signal=None, title=''):
    """Coefficient-wise posterior intervals vs truth and a point-estimate reference."""
    means = samples.mean(0); stds = samples.std(0)
    means_s = samples_s.mean(0); stds_s = samples_s.std(0)
    idx = np.arange(len(true_coefs))
    off = 0.18

    fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(idx) + 3), 4))
    ax.errorbar(idx - off, means, yerr=2 * stds, fmt='o', color='C0',
                label='Boomerang ±2σ', capsize=3, markersize=5)
    ax.errorbar(idx + off, means_s, yerr=2 * stds_s, fmt='s', color='C1',
                label='Sticky ±2σ', capsize=3, markersize=5)
    ax.scatter(idx, true_coefs, marker='x', color='k', s=70, linewidths=2,
               label='true', zorder=5)
    if ref_coefs is not None:
        ax.scatter(idx, ref_coefs, marker='_', color='grey', s=120, linewidths=2,
                   label='sklearn MLE', zorder=4)
    if is_signal is not None:
        for i in np.where(is_signal)[0]:
            ax.axvspan(i - 0.45, i + 0.45, color='gold', alpha=0.15)
    ax.axhline(0, color='grey', lw=0.5)
    ax.set_xticks(idx)
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('coefficient value')
    ax.set_title(title)
    ax.legend(loc='best', frameon=False, fontsize=8)
    plt.tight_layout()
    plt.show()

## 2. Multinomial logistic regression

3 classes, 6 features, coefficients drawn from N(0, 1.5²) — so the truth is *dense*: every coefficient is a signal. This is a different test: we're checking that the sampler recovers a non-sparse multinomial fit, not that sticky produces meaningful P(=0) values.

Note on flattening: the posterior is in whatever order `make_logistic_regression` uses internally. The cell below tries both row-major `(K, D+1)` and column-major `(D+1, K)` orderings against sklearn and reports the one that matches.

In [ ]:
rng = np.random.default_rng(0)
N, D, K = 300, 6, 3
X = rng.normal(size=(N, D))
beta_true_full = rng.normal(scale=1.5, size=(K, D + 1))

# Re-parameterise the TRUE coefficients in the class-0-reference frame
# by subtracting class 0's parameters from all K. The softmax likelihood
# is invariant under this, so the data-generating distribution is unchanged.
beta_true_ref = beta_true_full - beta_true_full[0:1]       # class 0 now exactly zero
beta_true_mat = beta_true_ref[1:]                           # keep only K-1 classes

# Generate data with the original full parameterisation
X_aug = np.hstack([np.ones((N, 1)), X])
logits_full = X_aug @ beta_true_full.T
probs = np.exp(logits_full - logits_full.max(1, keepdims=True))
probs /= probs.sum(1, keepdims=True)
y = np.array([rng.choice(K, p=p) for p in probs])
X = (X - X.mean(0)) / X.std(0)

# sklearn fits in this exact parameterisation internally
clf = LogisticRegression(C=np.inf, max_iter=2000).fit(X, y)
# sklearn gives coefs for all K classes but they sum to zero; take K-1
mle_mat = np.column_stack([clf.intercept_, clf.coef_])[1:]  # [K-1, D+1]

print(f"Multinomial (reference class 0): N={N}, K-1 fitted classes of shape {mle_mat.shape}")
print(f"Class counts: {np.bincount(y)}")

In [ ]:
# target = make_logistic_regression(
#     torch.tensor(X, dtype=torch.float64),
#     torch.tensor(y, dtype=torch.long),
#     prior_std=1.0, intercept_prior_std=10.0,
# )
import torch
K_fit = K - 1  # number of non-reference classes

X_t = torch.tensor(X, dtype=torch.float64)
y_t = torch.tensor(y, dtype=torch.long)
X_aug_t = torch.cat([torch.ones(N, 1, dtype=torch.float64), X_t], dim=1)

D_total = K_fit * (D + 1)
prec = torch.full((D_total,), 1.0, dtype=torch.float64)      # coefficient precision
# Intercepts at positions 0, (D+1), 2*(D+1), ... get loose prior
for k in range(K_fit):
    prec[k * (D + 1)] = 1.0 / 10.0**2

def energy_fn(beta):
    B = beta.reshape(K_fit, D + 1)
    logits_fit = X_aug_t @ B.T                                    # [N, K_fit]
    logits_full = torch.cat(
        [torch.zeros(N, 1, dtype=beta.dtype), logits_fit], dim=1  # [N, K]
    )
    neg_log_lik = torch.nn.functional.cross_entropy(
        logits_full, y_t, reduction="sum"
    )
    neg_log_prior = 0.5 * (prec * beta**2).sum()
    return neg_log_lik + neg_log_prior

def grad_target(beta):
    B = beta.reshape(K_fit, D + 1)
    logits_fit = X_aug_t @ B.T
    logits_full = torch.cat(
        [torch.zeros(N, 1, dtype=beta.dtype), logits_fit], dim=1
    )
    probs = torch.softmax(logits_full, dim=-1)                    # [N, K]
    y_oh = torch.zeros_like(probs)
    y_oh.scatter_(1, y_t.unsqueeze(1), 1.0)
    # Gradient only for the K_fit non-reference classes
    grad_logits_fit = (probs - y_oh)[:, 1:]                       # [N, K_fit]
    grad_B = X_aug_t.T @ grad_logits_fit                          # [D+1, K_fit]
    grad_lik = grad_B.T.reshape(-1)                               # flatten to [K_fit*(D+1)]
    return grad_lik + prec * beta

# Build a minimal target object
from sazz.models.smoke_test import TorchTarget
from sazz.utils.warmup import find_reference_glm

print(f"Finding reference (D_total={D_total})...")
x_ref, Sigma_inv = find_reference_glm(energy_fn, D_total, n_steps=2000, lr=1e-2, diagonal_only=False)

target = TorchTarget(
    name="logreg_multi_refclass0",
    D=D_total,
    grad_target=grad_target,
    x_ref=x_ref,
    Sigma_inv=Sigma_inv,
    meta={"n_classes": K, "K_fit": K_fit, "d_plus_1": D + 1},
)
# Automatic kappa (intercepts already handled large)
kappa = make_kappa_vector_glm(target.D, kappa_coef=1.0)
samples, samples_s = run_both(target, kappa, N_skel=30_000)
print(f"Posterior dimensions: {samples.shape[1]}  (expected {K * (D + 1)})")

In [ ]:
import pymc as pm

with pm.Model() as logreg_multi_model:
    intercepts = pm.Normal('intercepts', mu=0.0, sigma=10.0, shape=K_fit)
    betas = pm.Normal('betas', mu=0.0, sigma=1.0, shape=(K_fit, D))
    # Logits: shape [N, K] with class 0's logit = 0
    logits_fit = intercepts + X @ betas.T               # [N, K_fit]
    logits_full = pm.math.concatenate(
        [pm.math.zeros((N, 1)), logits_fit], axis=1
    )                                                    # [N, K]
    pm.Categorical('y', p=pm.math.softmax(logits_full, axis=-1), observed=y)

    nuts_multi = pm.sample(
        draws=2000, tune=1500, chains=2,
        target_accept=0.95, progressbar=True, random_seed=0,
    )

nuts_int = nuts_multi.posterior['intercepts'].values.reshape(-1, K_fit)
nuts_bet = nuts_multi.posterior['betas'].values.reshape(-1, K_fit, D)
nuts_mat = np.concatenate([nuts_int[:, :, None], nuts_bet], axis=-1)  # [S, K_fit, D+1]

In [ ]:
# Match whichever flattening the Boomerang posterior uses, by correlating
# the mean against both possibilities of the sklearn MLE
boom_means = samples.mean(0)
corr_row = np.corrcoef(boom_means, mle_mat.ravel())[0, 1]
corr_col = np.corrcoef(boom_means, mle_mat.T.ravel())[0, 1]

if corr_row >= corr_col:
    samples_nuts = nuts_mat.reshape(nuts_mat.shape[0], -1)                # row-major
    print(f"Row-major flattening (corr with MLE = {corr_row:.3f})")
else:
    samples_nuts = nuts_mat.transpose(0, 2, 1).reshape(nuts_mat.shape[0], -1)
    print(f"Column-major flattening (corr with MLE = {corr_col:.3f})")

print(f"NUTS (multinomial): {samples_nuts.shape}")

In [ ]:
means = samples.mean(0); stds = samples.std(0)
means_s = samples_s.mean(0); stds_s = samples_s.std(0)
means_n = samples_nuts.mean(0); stds_n = samples_nuts.std(0)
p_zero = (np.abs(samples_s) < 1e-8).mean(0)

# Flatten truth and MLE in the same order as the posterior
if corr_row >= corr_col:
    true_flat = beta_true_mat.ravel()
    mle_flat  = mle_mat.ravel()
    labels = [f'cls{k}_{"int." if i==0 else f"β_{i}"}'
              for k in range(K) for i in range(D + 1)]
else:
    true_flat = beta_true_mat.T.ravel()
    mle_flat  = mle_mat.T.ravel()
    labels = [f'{"int." if i==0 else f"β_{i}"}_cls{k}'
              for i in range(D + 1) for k in range(K)]

header = (f"{'coef':<16} {'true':>7} {'MLE':>7}"
          f"  |  {'NUTS μ':>7} {'NUTS σ':>7}"
          f"  |  {'Boom μ':>7} {'Boom σ':>7}"
          f"  |  {'Stky μ':>7} {'Stky σ':>7} {'P(=0)':>6}")
print(header); print('-' * len(header))
for i in range(len(true_flat)):
    print(f"{labels[i]:<16} {true_flat[i]:>7.3f} {mle_flat[i]:>7.3f}"
          f"  |  {means_n[i]:>7.3f} {stds_n[i]:>7.3f}"
          f"  |  {means[i]:>7.3f} {stds[i]:>7.3f}"
          f"  |  {means_s[i]:>7.3f} {stds_s[i]:>7.3f} {p_zero[i]:>6.2f}")

print(f"\nStd ratio (sampler/NUTS, averaged):")
print(f"  Boomerang: {(stds / stds_n).mean():.2f}  (want ≈ 1.0)"
      f"   Sticky: {(stds_s / stds_n).mean():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(max(10, 0.5 * len(true_flat) + 3), 4))
idx = np.arange(len(true_flat))
offset = 0.22

ax.errorbar(idx - offset, means_n, yerr=2 * stds_n, fmt='D', color='C2',
            label='NUTS ±2σ', capsize=3, markersize=5)
ax.errorbar(idx, means, yerr=2 * stds, fmt='o', color='C0',
            label='Boomerang ±2σ', capsize=3, markersize=5)
ax.errorbar(idx + offset, means_s, yerr=2 * stds_s, fmt='s', color='C1',
            label='Sticky ±2σ', capsize=3, markersize=5)
ax.scatter(idx, true_flat, marker='x', color='k', s=60, linewidths=2,
           label='true', zorder=5)

ax.axhline(0, color='grey', lw=0.5)
ax.set_xticks(idx)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('coefficient value')
ax.set_title('Multinomial logistic — every coefficient is a true signal')
ax.legend(loc='best', frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Pick the coefficient with largest true magnitude
j = np.argmax(np.abs(true_flat))

fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, samp, name, color in [
    (axes[0], samples_nuts, 'NUTS', 'C2'),
    (axes[1], samples, 'Boomerang', 'C0'),
    (axes[2], samples_s, 'Sticky', 'C1'),
]:
    ax.plot(samp[:, j], lw=0.4, color=color)
    ax.axhline(true_flat[j], color='k', lw=1, ls='--',
               label=f'true = {true_flat[j]:.2f}')
    ax.set_title(f'{name}: {labels[j]}')
    ax.set_xlabel('sample index')
    ax.legend(loc='best', fontsize=8, frameon=False)
axes[0].set_ylabel(labels[j])
plt.tight_layout(); plt.show()

In [ ]:
# Pick the coefficient with largest true magnitude
j = np.argmin(np.abs(true_flat))

fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, samp, name, color in [
    (axes[0], samples_nuts, 'NUTS', 'C2'),
    (axes[1], samples, 'Boomerang', 'C0'),
    (axes[2], samples_s, 'Sticky', 'C1'),
]:
    ax.plot(samp[:, j], lw=0.4, color=color)
    ax.axhline(true_flat[j], color='k', lw=1, ls='--',
               label=f'true = {true_flat[j]:.2f}')
    ax.set_title(f'{name}: {labels[j]}')
    ax.set_xlabel('sample index')
    ax.legend(loc='best', fontsize=8, frameon=False)
axes[0].set_ylabel(labels[j])
plt.tight_layout(); plt.show()